<a href="https://colab.research.google.com/github/chenharry66/thinkdsp-/blob/main/refactored_optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Setup

In [ ]:
#install depencies
!pip install werpy scipy
!pip install --upgrade "numba==0.68.0"
!pip install librosa

##Model Definition and Helper Functions

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import regex as re
import werpy, torch

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")

def custom_normalize(text):
    clean_text = re.sub(r'[\p{P}\p{S}]', '', text)
    return werpy.normalize(clean_text)

def load_file(path):
    rate, data = wavfile.read('path')
    data / 32768

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  151MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.75k [00:00<?, ?B/s]

In [ ]:

#save to some sort sort of location the file, but also metrics? wer  for exmaple
def save_result()

In [ ]:
def compute_log_spec(perturbation):
    window = torch.hann_window(400)
    stft = torch.stft(perturbation, 400, 160, window=window, return_complex=True)
    magnitudes = (stft[..., :-1].abs() ** 2).contiguous()
    mel_filters = torch.from_numpy(processor.feature_extractor.mel_filters).float()
    mel_spec = mel_filters.T @ magnitudes
    log_spec = torch.clamp(mel_spec, min=1e-10).log10()
    log_spec = torch.maximum(log_spec, log_spec.max() - 8.0)
    log_spec = (log_spec + 4.0) / 4.0

In [ ]:
# figure out how the

def training_1(MAX_ITERS=200, ALPHA=1e-5, clip, labels):
    audio_rms = np.float32(np.sqrt(np.mean(clip ** 2)))
    target_rms = audio_rms/(10**(23/20))
    delta = torch.zeros_like(clip, requires_grad=True)
    clip=torch.tensor(clip)

    for p in model.parameters():
        p.requires_grad = False

    for i in range(MAX_ITERS):
        delta.grad = None
        perturbation = clip + delta
        log_spec = compute_log_spec_perturation
        out = model(log_spec.unsqueeze(0), labels=labels, return_tensors="pt")
        loss = out.loss
        print(f"Iteration {i} Loss: {loss.item()}")
        loss.backward()
        with torch.no_grad():
            delta += ALPHA * delta.grad
            current_rms = torch.sqrt(torch.mean(delta**2)) + 1e-6
            print(f"Iteration {i} RMS: {current_rms}")
            if current_rms > target_rms:
                delta *= (target_rms / current_rms)
    return delta

In [ ]:
# returns a wer
def eval(aug_audio, reference)
    inputs = processor(aug_audio, return_tensors="pt", sampling_rate=rate)
    generated_ids = model.generate(input_features=inputs['input_features'])
    transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    transcription = custom_normalize(transcription)
    wer = werpy.wer(reference, transcription)

In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/FOLDER/01")
import generate_masking_threshold as masking
theta_xs, psd_max = masking.generate_th(clip, rate, window_size=2048)

In [ ]:
import librosa

def compute_PSD_for_perturbation(perturbation, psd_max, window_size):
    win = torch.hann_window(window_size)
    spectrum = torch.stft(perturbation, window=win, n_fft = window_size, hop_length=window_size//4, onesided=True, center=False, return_complex=True)
    z = torch.sqrt(torch.tensor(8 / 3)) * torch.abs(spectrum / window_size)
    psd = 10 * torch.log10(z * z + 0.0000000000000000001)
    PSD = 96 - 10 * torch.log10(psd_max + 0.0000000000000000001) + psd
    PSD = 10**(PSD/10)
    return PSD.T

In [ ]:
def compute_mask_threshold_loss(psd_perturb, theta_xs):
    average_loss_per_frame = torch.mean((psd_perturb - theta_xs).relu(), dim=1)
    return torch.mean(average_loss_per_frame)

In [ ]:
def training2(clip, delta, MAX_ITERS=200, ALPHA=1e-5, BETA=1e-1):
    clip = torch.tensor(clip)
    theta_xs = torch.tensor(theta_xs)
    window_size = 2048
    psd_max = torch.tensor(psd_max)

    for p in model.parameters():
        p.requires_grad = False

    for i in range(MAX_ITERS):
        delta.grad=None
        perturbation = clip + delta
        compute_log_spec(perturbation)
        out = model(log_spec.unsqueeze(0), labels=labels, return_tensors="pt")
        psd_perturb = compute_PSD_for_perturbation(delta, psd_max, window_size)
        loss = out.loss - BETA * compute_mask_threshold_loss(psd_perturb, theta_xs)
        print(f"Iteration {i} Loss: {loss.item()}")
        loss.backward()
        with torch.no_grad():
            delta += ALPHA * delta.grad
    return delta

In [ ]:
def optimize_one_recording(clip, reference):
    labels = processor.tokenizer(reference, return_tensors="pt", return_attention_mask=False)['input_ids']



